# Entraînement final et exploration du modèle BERTopic

Ce notebook recharge les paramètres retenus lors de l'optimisation, entraîne le modèle final et explore la structure thématique du corpus.

In [1]:
import gc
import time
import warnings
from itertools import combinations
from pathlib import Path #pour la gestion des chemins de fichiers
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP
from hdbscan import HDBSCAN
from hdbscan.validity import validity_index

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer #pour les embeddings de phrase

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

COL_TEXTE = "phrases_lemm"
COL_ROMAN = "roman"

RANDOM_STATE = 42


warnings.filterwarnings("ignore")

## 1. Chargement du corpus et chargement ou calcul des embeddings

In [2]:
df=pd.read_csv(Path("..")/"data" /"2_processed"/"03_corpus_lematise.csv", encoding="utf-8")

CHEMIN_EMBEDDINGS = Path("../data/donnees_annex/embeddings_sentence_camembert.npy")

if CHEMIN_EMBEDDINGS.exists():
    print("Chargement des embeddings sauvegardés...")
    embeddings = np.load(CHEMIN_EMBEDDINGS, allow_pickle=False)
else:
    embedding_model = SentenceTransformer(
        "dangvantuan/sentence-camembert-base"
    )

    print("Génération des embeddings sémantiques...")
    embeddings = embedding_model.encode(
        df["texte"].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    CHEMIN_EMBEDDINGS.parent.mkdir(parents=True, exist_ok=True)
    np.save(CHEMIN_EMBEDDINGS, embeddings)
    print(f"Embeddings sauvegardés dans : {CHEMIN_EMBEDDINGS}")

Chargement des embeddings sauvegardés...


## 2. Préparation des documents pour BERTopic

In [3]:
# Vérification des colonnes
assert COL_TEXTE in df.columns, (
    f"La colonne '{COL_TEXTE}' n'existe pas dans df."
)
assert COL_ROMAN in df.columns, (
    f"La colonne '{COL_ROMAN}' n'existe pas dans df."
)

# Conversion des embeddings
embeddings_array_initial = np.asarray(embeddings)

assert len(df) == len(embeddings_array_initial), (
    "Le nombre de lignes de df ne correspond pas "
    "au nombre d'embeddings."
)

# Masque des documents non vides
masque_documents = (df[COL_TEXTE].notna() & df[COL_TEXTE].astype(str).str.strip().ne(""))

# DataFrame utilisé par BERTopic
df_model = (df.loc[masque_documents].copy().reset_index(drop=True))

# Documents
documents = (df_model[COL_TEXTE].astype(str).tolist())

# Romans associés aux documents
romans = (df_model[COL_ROMAN].astype(str).tolist())

# Embeddings alignés
embeddings_array = embeddings_array_initial[masque_documents.to_numpy()]

# Tokenisation simple pour la cohérence C_v
texts_tokenises = [
    document.split()
    for document in documents
]

# Dictionnaire Gensim
dictionary = Dictionary(texts_tokenises)


print("Nombre de documents :", len(documents))
print("Nombre de romans :", df_model[COL_ROMAN].nunique())
print("Dimensions des embeddings :", embeddings_array.shape)
print("Taille du dictionnaire :", len(dictionary))

Nombre de documents : 5327
Nombre de romans : 31
Dimensions des embeddings : (5327, 768)
Taille du dictionnaire : 19186


## 3. Fonctions d'évaluation finale

In [4]:
def extraire_mots_topics(
    topic_model,
    labels,
    top_n_words=10,
    dictionary=None
):
    """
    Extrait les mots des topics, en excluant le topic -1.

    Si un dictionnaire Gensim est fourni, seuls les mots présents
    dans ce dictionnaire sont conservés.
    """

    labels = np.asarray(labels)

    topic_ids = sorted(
        int(topic_id)
        for topic_id in np.unique(labels)
        if topic_id != -1
    )

    topics_words = []

    for topic_id in topic_ids:

        representation = topic_model.get_topic(topic_id)

        if not representation:
            continue

        words = [
            word
            for word, _ in representation[:top_n_words]
        ]

        if dictionary is not None:
            words = [
                word
                for word in words
                if word in dictionary.token2id
            ]

        # Éviter les topics insuffisamment représentés
        if len(words) >= 2:
            topics_words.append(words)

    return topics_words


def calculer_coherence_cv(
    topic_model,
    labels,
    texts_tokenises,
    dictionary,
    top_n_words=10
):
    """
    Calcule la cohérence C_v des topics BERTopic.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=dictionary
    )

    if len(topics_words) < 2:
        return np.nan

    try:
        coherence_model = CoherenceModel(
            topics=topics_words,
            texts=texts_tokenises,
            dictionary=dictionary,
            coherence="c_v",
            topn=top_n_words,
            processes=1
        )

        return float(coherence_model.get_coherence())

    except Exception:
        return np.nan

def calculer_diversite_topics(
    topic_model,
    labels,
    top_n_words=10
):
    """
    Diversité lexicale :
    nombre de termes uniques / nombre total de termes.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=None
    )

    tous_les_mots = [
        word
        for topic_words in topics_words
        for word in topic_words
    ]

    if not tous_les_mots:
        return np.nan

    return len(set(tous_les_mots)) / len(tous_les_mots)
    

## 4. Chargement des paramètres et entraînement du modèle final

Les paramètres sélectionnés dans le notebook d'optimisation sont rechargés depuis le fichier JSON.

In [5]:
CHEMIN_PARAMETRES = Path("../data/donnees_annex/meilleurs_parametres_bertopic.json")

with CHEMIN_PARAMETRES.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

best_umap_model = UMAP(
    n_neighbors=best_params["n_neighbors"],
    n_components=best_params["n_components"],
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=True
)

best_hdbscan_model = HDBSCAN(
    min_cluster_size=best_params["min_cluster_size"],
    min_samples=best_params["min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
    core_dist_n_jobs=-1
)

best_vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.8,
)

best_ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model = BERTopic(
    language="french",
    hdbscan_model=best_hdbscan_model,
    umap_model=best_umap_model,
    vectorizer_model=best_vectorizer_model,
    ctfidf_model=best_ctfidf_model,
    calculate_probabilities=False,
    nr_topics=None,
    verbose=True
)

best_topics_raw, _ = best_topic_model.fit_transform(
    documents,
    embeddings=embeddings_array
)

best_topics_raw = np.asarray(best_topics_raw)


nombre_topics_bruts = len(
    np.unique(
        best_topics_raw[
            best_topics_raw != -1
        ]
    )
)

taux_outliers_brut = np.mean(best_topics_raw == -1)

print("Nombre naturel de topics :", nombre_topics_bruts)

print( f"Taux brut d'outliers : " f"{taux_outliers_brut:.1%}")

display(best_topic_model.get_topic_info())

2026-07-29 23:37:45,132 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2026-07-29 23:37:55,202 - BERTopic - Dimensionality - Completed ✓
2026-07-29 23:37:55,203 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-29 23:37:55,340 - BERTopic - Cluster - Completed ✓
2026-07-29 23:37:55,342 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-29 23:37:55,548 - BERTopic - Representation - Completed ✓


Nombre naturel de topics : 33
Taux brut d'outliers : 55.6%


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2961,-1_délicieux_usine_village_rideau,"[délicieux, usine, village, rideau, curé, sièc...",[position quartier ménagement peur place honne...
1,0,268,0_étreinte_chéri_veu_coupable,"[étreinte, chéri, veu, coupable, embrass, mère...",[bois pain fin éveillé jardin but récompense g...
2,1,252,1_époux_volupté_noiraud_étreinte,"[époux, volupté, noiraud, étreinte, nerf, effr...",[approche plein angoisse contentement désir se...
3,2,130,2_lac_verdure_toiture_nef,"[lac, verdure, toiture, nef, feuillage, rive, ...",[porte ouvert voûte profond cours intérieur hu...
4,3,130,3_social_catholicisme_humanité_religion,"[social, catholicisme, humanité, religion, nat...",[enseignement intégral école primaire laïque o...
5,4,115,4_cardinal_pape_évêque_curé,"[cardinal, pape, évêque, curé, ministre, préla...",[mesure long attente grâce prêtre faute -vou é...
6,5,99,5_usurier_barbue_billet_crédit,"[usurier, barbue, billet, crédit, concession, ...",[nouveau juge procès intervention apparence mo...
7,6,97,6_instituteur_spéculation_baron_dot,"[instituteur, spéculation, baron, dot, créanci...",[déconvenue visite personne influent frère cli...
8,7,96,7_dite_vœu_râle_mademoiselle,"[dite, vœu, râle, mademoiselle, meneur, divin,...",[incroyable peine temporel exemple question la...
9,8,91,8_infirmier_blessé_canotier_cuisse,"[infirmier, blessé, canotier, cuisse, veau, pa...",[inutile trou pont pont écroulement rivière gr...


## 5. Réaffectation des documents hors topic

Plusieurs seuils sont comparés avant de retenir le seuil final de réaffectation des outliers.

In [6]:
seuils_outliers = [
    0.0,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

comparaisons_seuils = []
topics_par_seuil = {}

for seuil in seuils_outliers:
    topics_test = best_topic_model.reduce_outliers(
        documents,
        best_topics_raw,
        strategy="embeddings",
        embeddings=embeddings_array,
        threshold=seuil)

    topics_test = np.asarray(topics_test)

    topics_par_seuil[seuil] = topics_test

    masque_assignes = topics_test != -1

    tailles_topics = (pd.Series(topics_test[masque_assignes]).value_counts())

    nombre_assignes = int(masque_assignes.sum())

    nombre_reassignes = int(((best_topics_raw == -1) & (topics_test != -1)).sum())

    comparaisons_seuils.append({
        "threshold": seuil,
        "n_topics": len(tailles_topics),
        "outlier_rate_remaining": np.mean(
            topics_test == -1
        ),
        "n_reassigned": nombre_reassignes,
        "smallest_topic": (
            tailles_topics.min()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic": (
            tailles_topics.max()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic_share": (
            tailles_topics.max() / nombre_assignes
            if nombre_assignes > 0
            else np.nan
        )
    })


comparaison_seuils_df = pd.DataFrame(comparaisons_seuils)

display(
    comparaison_seuils_df.style.format({
        "threshold": "{:.2f}",
        "outlier_rate_remaining": "{:.1%}",
        "largest_topic_share": "{:.1%}"
    })
)

,threshold,n_topics,outlier_rate_remaining,n_reassigned,smallest_topic,largest_topic,largest_topic_share
0,0.00,33,0.0%,2961,38,422,7.9%
1,0.10,33,0.0%,2961,38,422,7.9%
2,0.20,33,0.0%,2961,38,422,7.9%
3,0.30,33,0.0%,2961,38,422,7.9%
4,0.40,33,0.0%,2961,38,422,7.9%
5,0.50,33,0.0%,2960,38,422,7.9%


In [7]:
SEUIL_OUTLIERS_FINAL = 0.30

topics_finaux = np.asarray(topics_par_seuil[SEUIL_OUTLIERS_FINAL])

print(
    f"Taux d'outliers final : "
    f"{np.mean(topics_finaux == -1):.1%}"
)


print(
    "Nombre final de topics :",
    len(np.unique(topics_finaux[topics_finaux != -1])))

distribution_topics = (
    pd.Series(topics_finaux)
    .value_counts()
    .sort_index()
    .rename_axis("Topic")
    .reset_index(name="Count")
)

display(distribution_topics)

Taux d'outliers final : 0.0%
Nombre final de topics : 33


,Topic,Count
0,0,345
1,1,422
2,2,198
3,3,205
4,4,216
5,5,199
6,6,226
7,7,162
8,8,212
9,9,207


## 6. Raffinement de la représentation lexicale des topics

In [8]:
stopwords_corpus = ["deberl", "men", "yole","embrass", "revien", "tai", "connai", "quidquid", "trouche", "hattoy"]

vectorizer_final = CountVectorizer(
    stop_words=stopwords_corpus,
    min_df=2,
    max_df=0.80,
    #ngram_range=(1, 2)
)

ctfidf_final = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model.update_topics(
    documents,
    topics=topics_finaux,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    top_n_words=10
)

display(best_topic_model.get_topic_info())

2026-07-29 23:37:55,778 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,0,345,0_pleur_répond_balbutiant_infâme,"[pleur, répond, balbutiant, infâme, pleure, pe...",[bois pain fin éveillé jardin but récompense g...
1,1,422,1_noiraud_nerf_déchirement_maternité,"[noiraud, nerf, déchirement, maternité, inerte...",[approche plein angoisse contentement désir se...
2,2,198,2_tombeau_temple_pilier_mont,"[tombeau, temple, pilier, mont, statue, rive, ...",[porte ouvert voûte profond cours intérieur hu...
3,3,205,3_nation_catholicisme_salariat_socialisme,"[nation, catholicisme, salariat, socialisme, a...",[enseignement intégral école primaire laïque o...
4,4,216,4_cardinal_jésuite_vicaire_prélat,"[cardinal, jésuite, vicaire, prélat, eminence,...",[mesure long attente grâce prêtre faute -vou é...
5,5,199,5_usurier_bonapartiste_barbue_armateur,"[usurier, bonapartiste, barbue, armateur, spéc...",[nouveau juge procès intervention apparence mo...
6,6,226,6_hérédité_dot_baron_loyer,"[hérédité, dot, baron, loyer, spéculation, ind...",[déconvenue visite personne influent frère cli...
7,7,162,7_bouquetière_antéchrist_vœu_voiturier,"[bouquetière, antéchrist, vœu, voiturier, impu...",[incroyable peine temporel exemple question la...
8,8,212,8_infirmier_veau_canotier_flocon,"[infirmier, veau, canotier, flocon, contractio...",[inutile trou pont pont écroulement rivière gr...
9,9,207,9_ambulance_robinet_professeur_pelisse,"[ambulance, robinet, professeur, pelisse, quen...",[mariage démarche semaine question banquette t...


## 7. Évaluation du modèle final

In [9]:
# L'analyseur produit exactement les tokens 
# attendus par le nouveau CountVectorizer.

analyseur_final = (vectorizer_final.build_analyzer())

texts_tokenises_final = [
    analyseur_final(document)
    for document in documents
]


dictionary_final = Dictionary(texts_tokenises_final)


coherence_finale = calculer_coherence_cv(
    topic_model=best_topic_model,
    labels=topics_finaux,
    texts_tokenises=texts_tokenises_final,
    dictionary=dictionary_final,
    top_n_words=10
)


diversite_finale = calculer_diversite_topics(
    topic_model=best_topic_model,
    labels=topics_finaux,
    top_n_words=10
)


print(
    f"Cohérence C_v finale : "
    f"{coherence_finale:.3f}"
)

print(
    f"Diversité finale : "
    f"{diversite_finale:.3f}"
)

Cohérence C_v finale : 0.471
Diversité finale : 0.915


## 8. Visualisations thématiques et temporelles

In [10]:
fig = best_topic_model.visualize_hierarchy()
fig.show()

In [21]:
# Calcul de l'évolution temporelle
topics_over_time = best_topic_model.topics_over_time(
    documents,
    df_model["annee"].tolist(),
    nr_bins=15
)

# Noms plus lisibles

# Création du graphique
fig = best_topic_model.visualize_topics_over_time(
    topics_over_time,
    custom_labels=True,
    normalize_frequency=False,
    title="Travail et milieux populaires",
    width=1500,
    height=700
)

# Personnalisation en français
fig.update_traces(mode="lines+markers")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Période de publication",
    yaxis_title="Nombre de segments",
    legend_title_text=None
)



15it [00:00, 15.78it/s]


## 9. Analyse de la distribution des topics par roman

In [ ]:
df_resultats = df_model.copy()

df_resultats["topic"] = topics_finaux

df_resultats["est_outlier"] = (df_resultats["topic"] == -1)

display(df_resultats[[COL_ROMAN, COL_TEXTE, "topic", "est_outlier"]].head())

,roman,phrases_lemm,topic,est_outlier
0,1865 La confession de Claude.,hiver matin frais manteau brouillard saison so...,32,False
1,1865 La confession de Claude.,soif projet soi ferme loyal action rêve vis gr...,0,False
2,1865 La confession de Claude.,rayon cauchemar veille obstacle lutte désert c...,32,False
3,1865 La confession de Claude.,ombre réalité inquiet joie conscience vague pl...,32,False
4,1865 La confession de Claude.,corps horrible rapide impression dégoût effroi...,1,False


In [ ]:
table_topics_romans_counts = pd.crosstab(df_resultats[COL_ROMAN],df_resultats["topic"])

display(table_topics_romans_counts)

topic,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,32
roman,,,,,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,17,4,2,0,0,0,0,2,5,1,...,0,0,0,2,0,1,0,0,1,27
1866 Le voeu d une morte.,5,11,2,1,0,2,1,7,1,3,...,1,4,2,1,7,1,0,0,2,0
1867 Les mysteres de Marseille.,6,6,2,3,24,27,10,11,9,3,...,0,11,2,0,2,2,0,7,2,0
1867 Therese Raquin.,8,17,3,0,2,1,2,4,11,4,...,0,4,5,0,0,1,1,1,2,0
1868 Madeleine Ferat.,16,40,5,1,3,1,6,6,1,7,...,1,6,2,0,5,1,1,1,3,3
Au Bonheur des dames.,4,10,5,0,1,6,11,5,3,4,...,0,2,3,7,6,11,6,9,0,0
Fecondite.,13,27,5,16,5,8,18,15,6,9,...,17,11,4,2,12,0,1,6,3,3
Germinal.,9,10,6,13,4,10,7,1,12,2,...,3,1,10,1,3,2,0,6,2,0
L argent.,3,3,2,6,7,11,7,6,3,1,...,6,5,3,1,2,1,24,1,0,0


In [ ]:
table_topics_romans_proportions = pd.crosstab(
    df_resultats[COL_ROMAN],
    df_resultats["topic"],
    normalize="index"
)

display(table_topics_romans_proportions.style.format("{:.1%}"))

topic,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32
roman,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,26.2%,6.2%,3.1%,0.0%,0.0%,0.0%,0.0%,3.1%,7.7%,1.5%,0.0%,0.0%,1.5%,3.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,3.1%,0.0%,1.5%,0.0%,0.0%,1.5%,41.5%
1866 Le voeu d une morte.,8.3%,18.3%,3.3%,1.7%,0.0%,3.3%,1.7%,11.7%,1.7%,5.0%,1.7%,0.0%,0.0%,3.3%,0.0%,3.3%,0.0%,3.3%,1.7%,0.0%,0.0%,0.0%,1.7%,1.7%,6.7%,3.3%,1.7%,11.7%,1.7%,0.0%,0.0%,3.3%,0.0%
1867 Les mysteres de Marseille.,3.3%,3.3%,1.1%,1.6%,13.0%,14.7%,5.4%,6.0%,4.9%,1.6%,12.5%,1.1%,0.5%,1.1%,0.5%,4.3%,0.5%,1.6%,1.6%,2.7%,1.6%,2.7%,0.0%,0.0%,6.0%,1.1%,0.0%,1.1%,1.1%,0.0%,3.8%,1.1%,0.0%
1867 Therese Raquin.,10.0%,21.2%,3.8%,0.0%,2.5%,1.2%,2.5%,5.0%,13.8%,5.0%,0.0%,0.0%,3.8%,1.2%,1.2%,3.8%,0.0%,0.0%,0.0%,1.2%,0.0%,3.8%,2.5%,0.0%,5.0%,6.2%,0.0%,0.0%,1.2%,1.2%,1.2%,2.5%,0.0%
1868 Madeleine Ferat.,12.9%,32.3%,4.0%,0.8%,2.4%,0.8%,4.8%,4.8%,0.8%,5.6%,0.8%,0.8%,1.6%,1.6%,0.8%,1.6%,0.0%,0.0%,0.0%,3.2%,0.0%,1.6%,0.0%,0.8%,4.8%,1.6%,0.0%,4.0%,0.8%,0.8%,0.8%,2.4%,2.4%
Au Bonheur des dames.,2.0%,5.1%,2.5%,0.0%,0.5%,3.0%,5.6%,2.5%,1.5%,2.0%,0.5%,8.6%,0.5%,4.5%,3.5%,2.0%,7.1%,8.1%,3.5%,6.6%,0.0%,1.0%,7.1%,0.0%,1.0%,1.5%,3.5%,3.0%,5.6%,3.0%,4.5%,0.0%,0.0%
Fecondite.,5.0%,10.4%,1.9%,6.2%,1.9%,3.1%,6.9%,5.8%,2.3%,3.5%,0.0%,1.2%,1.9%,8.1%,1.9%,1.9%,1.9%,5.8%,1.9%,1.9%,1.5%,1.2%,0.8%,6.6%,4.2%,1.5%,0.8%,4.6%,0.0%,0.4%,2.3%,1.2%,1.2%
Germinal.,4.0%,4.4%,2.7%,5.8%,1.8%,4.4%,3.1%,0.4%,5.3%,0.9%,2.2%,9.3%,5.3%,0.9%,14.7%,5.8%,2.7%,0.4%,8.0%,1.3%,0.0%,1.8%,2.2%,1.3%,0.4%,4.4%,0.4%,1.3%,0.9%,0.0%,2.7%,0.9%,0.0%
L argent.,1.9%,1.9%,1.3%,3.9%,4.5%,7.1%,4.5%,3.9%,1.9%,0.6%,0.0%,5.2%,0.6%,6.5%,2.6%,5.2%,3.2%,1.9%,3.9%,6.5%,1.9%,0.6%,2.6%,3.9%,3.2%,1.9%,0.6%,1.3%,0.6%,15.5%,0.6%,0.0%,0.0%


In [ ]:
topics_par_roman = (
    best_topic_model
    .topics_per_class(
        documents,
        classes=romans,
        global_tuning=True
    )
)

display(topics_par_roman)

31it [00:00, 60.85it/s]


,Topic,Words,Frequency,Class
0,0,"courtier, capuchon, domino, bossu, frison",14,La curee.
1,1,"serre, inceste, calèche, bracelet, aigrette",12,La curee.
2,2,"calèche, taillis, socle, boudeur, harnais",14,La curee.
3,3,"parussent, parenté, lampiste, fouine, législatif",1,La curee.
4,4,"compère, entrepositaire, massacrant, baron, pé...",2,La curee.
...,...,...,...,...
840,26,"grêlon, soupesait, mousqueterie, phosphore, mi...",1,La terre.
841,27,"confiserie, culotté, incolore, chieur, brûlot",2,La terre.
842,28,"déshonoraient, moutarde, merde, pigeonnier, ér...",1,La terre.
843,30,"trombone, conscrit, piston, chantre, cercueil",3,La terre.


## 10. Export des résultats

In [ ]:
chemin_sortie = Path("..") /"data" /"4_resultats" /"documents_avec_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
df_resultats.to_csv(chemin_sortie, index=False, encoding="utf-8")


chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_comptages.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_counts.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_proportions.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_proportions.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"bertopic_topics_per_class.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
topics_par_roman.to_csv(chemin_sortie, index=False, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"informations_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
best_topic_model.get_topic_info().to_csv(chemin_sortie, index=False, encoding="utf-8")




Résultats sauvegardés dans le dossier :  ../data/4_resultats
